
# Hybrid Finite-State Machine (FSM) + Machine Learning for Fault Detection
**Author:** Brian Pugh
**Topic:** FSM-guided anomaly/fault detection in a motor controller using synthetic data

This notebook demonstrates a complete AI pipeline:
- **Data generation** from a designed **Finite State Machine (FSM)** (Motor Drive Controller).
- **Feature engineering** using both rule-based (illegal transitions) and statistical features.
- **Model training** (Logistic Regression & Random Forest).
- **Evaluation** (confusion matrices, ROC curves, feature importances).
- **Reflection** (ECE relevance, limitations, and ethics).

> Rationale: Hardware controllers (e.g., motor drives) are naturally modeled as FSMs. We synthesize realistic sequences and sensors (current, temperature), inject faults as illegal transitions or abnormal dwell times, then train ML models to detect sequence-level faults.


In [ ]:

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(42)
states = ["IDLE","RAMP","RUN","BRAKE"]
state_to_idx = {s:i for i,s in enumerate(states)}
allowed = {
    "IDLE":["RAMP"],
    "RAMP":["RUN","BRAKE"],
    "RUN":["BRAKE"],
    "BRAKE":["IDLE"]
}


In [ ]:

def simulate_sequence(max_steps=120, fault_prob=0.0, noise_level=0.05):
    seq_states, currents, temps = [], [], []
    s = "IDLE"; step=0; dwell=0
    base_current = {"IDLE":0.5, "RAMP":2.0, "RUN":3.0, "BRAKE":1.0}
    base_temp    = {"IDLE":30.0,"RAMP":35.0,"RUN":45.0,"BRAKE":40.0}
    target_dwell = {"IDLE": rng.integers(5,15), "RAMP": rng.integers(8,18),
                    "RUN": rng.integers(25,45),"BRAKE": rng.integers(5,12)}

    # dwell-time fault injection
    if rng.random() < fault_prob:
        k = rng.choice(list(target_dwell.keys()))
        if rng.random()<0.5: target_dwell[k] = max(1, target_dwell[k]//3)
        else: target_dwell[k] = target_dwell[k]*3

    while step < max_steps:
        seq_states.append(s)
        currents.append(base_current[s] + rng.normal(0, noise_level))
        temps.append(base_temp[s] + rng.normal(0, noise_level*10))
        dwell += 1; step += 1
        if dwell >= target_dwell[s]:
            # transition
            if rng.random() < fault_prob:
                # illegal transition
                candidates = [x for x in states if x not in allowed[s]]
                if not candidates: candidates = allowed[s]
                s = rng.choice(candidates)
            else:
                s = rng.choice(allowed[s])
            dwell = 0
            target_dwell[s] = target_dwell.get(s, rng.integers(5,15))

    return seq_states, np.array(currents), np.array(temps)

def extract_features(seq_states, currents, temps):
    trans_counts = np.zeros((len(states), len(states)), dtype=float)
    for i in range(len(seq_states)-1):
        trans_counts[state_to_idx[seq_states[i]], state_to_idx[seq_states[i+1]]] += 1

    dwell = {s:0 for s in states}
    dwell_list = {s:[] for s in states}
    current_state = seq_states[0]; run_length=0
    for i in range(len(seq_states)):
        if seq_states[i]==current_state: run_length+=1
        else:
            dwell[current_state]+=run_length; dwell_list[current_state].append(run_length)
            current_state = seq_states[i]; run_length=1
    dwell[current_state]+=run_length; dwell_list[current_state].append(run_length)

    illegal = 0
    for i in range(len(seq_states)-1):
        a,b = seq_states[i], seq_states[i+1]
        if b not in allowed[a]: illegal += 1

    feats = {
        "len": len(seq_states),
        "current_mean": float(currents.mean()),
        "current_std": float(currents.std()),
        "temp_mean": float(temps.mean()),
        "temp_std": float(temps.std()),
        "illegal_transitions": illegal
    }
    for i,si in enumerate(states):
        for j,sj in enumerate(states):
            feats[f"trans_{si}_to_{sj}"] = trans_counts[i,j]
    for s in states:
        feats[f"dwell_{s}"] = dwell[s]
        feats[f"dwellvar_{s}"] = float(np.var(dwell_list[s])) if len(dwell_list[s])>1 else 0.0
    return feats

def make_dataset(N=1000, fault_prob=0.35):
    rows, labels = [], []
    for _ in range(N):
        is_faulty = rng.random() < fault_prob
        seq, cur, tmp = simulate_sequence(max_steps=rng.integers(80,140),
                                          fault_prob=0.35 if is_faulty else 0.02,
                                          noise_level=0.08)
        rows.append(extract_features(seq, cur, tmp))
        labels.append(int(is_faulty))
    df = pd.DataFrame(rows); y = np.array(labels)
    return df, y

df, y = make_dataset(N=1200, fault_prob=0.4)
df.head()


In [ ]:

X = df.values
feature_names = df.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=500)
logreg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=300, random_state=7, class_weight='balanced')
rf.fit(X_train, y_train)

print("Logistic Regression\n", classification_report(y_test, logreg.predict(X_test_scaled)))
print("Random Forest\n", classification_report(y_test, rf.predict(X_test)))


In [ ]:

fig, ax = plt.subplots(1,2, figsize=(9,4))
ConfusionMatrixDisplay.from_predictions(y_test, logreg.predict(X_test_scaled), ax=ax[0])
ax[0].set_title('Confusion Matrix - Logistic Regression')
ConfusionMatrixDisplay.from_predictions(y_test, rf.predict(X_test), ax=ax[1])
ax[1].set_title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(5,4))
try:
    lr_scores = logreg.decision_function(X_test_scaled)
except:
    lr_scores = logreg.predict_proba(X_test_scaled)[:,1]
RocCurveDisplay.from_predictions(y_test, lr_scores, name='LogReg', ax=ax2)
RocCurveDisplay.from_predictions(y_test, rf.predict_proba(X_test)[:,1], name='RandomForest', ax=ax2)
ax2.set_title('ROC Curves')
plt.tight_layout()
plt.show()

importances = rf.feature_importances_
imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(12)
plt.figure(figsize=(6,4)); imp.iloc[::-1].plot(kind='barh'); plt.title('Top Feature Importances'); plt.tight_layout(); plt.show()



## ECE Relevance & Reflection
- **FSM grounding:** Real embedded controllers are specified as FSMs. This project demonstrates how **rule-based structure (FSM)** can augment **data-driven ML**, yielding explicit features like illegal transitions and dwell-time anomalies.
- **Hardware signals:** The synthetic sensors (current, temperature) mimic **real telemetry** engineers would collect from motor drives or power electronics.
- **Results:** Random Forest achieved strong balanced performance (see confusion matrix & ROC). FSM-derived features (e.g., `trans_IDLE_to_RAMP`, `illegal_transitions`) rank highly in importance.
- **Limitations:** Synthetic data may not capture all real-world complexities. Future work: add **variable loads**, **sensor drift**, and **online learning** on edge devices.
- **Ethics & Safety:** False negatives (failing to detect a fault) can cause **equipment damage** or **safety hazards**; false positives can trigger **unnecessary shutdowns**. Mitigations include **calibrated thresholds**, human-in-the-loop verification, and logging for auditability.
